In [1]:
import json
from datetime import datetime
import re

# 1. 读取JSON文件
with open('result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. 查看数据结构
print(f"总帖子数: {len(data)}")
print(f"\n第一条帖子的键: {data[0].keys()}")
print(f"\ndetail里的键: {data[0]['detail'].keys()}")

总帖子数: 3401

第一条帖子的键: dict_keys(['post_id', 'detail', 'comments'])

detail里的键: dict_keys(['PostID', 'UserID', 'UserName', 'UserScore', 'UserTelephone', 'UserAvatar', 'UserIdentity', 'Title', 'Content', 'Like', 'Comment', 'Browse', 'Heat', 'PostTime', 'IsSaved', 'IsLiked', 'Photos', 'Tag'])


处理时间

In [5]:
def format_timestamp(iso_time):
    """将ISO时间转换为中文可读格式"""
    if not iso_time:
        return ""
    try:
        # 去掉时区信息，解析时间
        dt = datetime.fromisoformat(iso_time.replace('+08:00', ''))
        # 格式化为：2026年3月26日
        return dt.strftime('%Y年%m月%d日')
    except:
        return iso_time

# 测试
test_time = "2026-03-26T12:29:53+08:00"
print(format_timestamp(test_time))  # 输出: 2026年03月26日

2026年03月26日


处理评论

In [11]:
def extract_all_comments_flat(comments):
    """
    提取所有评论，完全扁平化，不区分主评论/子评论
    所有评论同等地位，同属一个帖子
    """
    all_comments = []
    
    def _collect(comms):
        """递归收集所有评论，不管层级"""
        for comment in comms:
            # 提取作者和内容（兼容大小写）
            author = comment.get('Author') or comment.get('author', '')
            content = comment.get('Content') or comment.get('content', '')
            content = content.strip() if content else ''
            
            if author and content:
                # 统一格式，不标记层级
                all_comments.append(f"{author}: {content}")
            
            # 继续递归收集子评论（如果有的话）
            sub_comments = comment.get('SubComments', [])
            if sub_comments:
                _collect(sub_comments)  # 不增加缩进，不传递层级信息
    
    _collect(comments)
    return all_comments


# ========== 测试 ==========
first_post = data[9]
comments = first_post.get('comments', [])

print("扁平化后的所有评论：")
result = extract_all_comments_flat(comments)
for i, line in enumerate(result, 1):
    print(f"{i}. {line}")

print(f"\n共 {len(result)} 条评论（主+子都混在一起）")

扁平化后的所有评论：
1. 肖迪种地and大敌: 应该得先在企微的“大学服务中心”申请补办校园卡，然后等通知到瀚林1号领。不过不用校园卡，直接用企微的热水服务也可以用热水的吧
2. 阿平平平安安: 谢谢uu🥰你是一个很好很好的宝宝
3. N/A: 那也得看他当初有没有把卡绑到过手机上
4. 十栖: 珠海校区补办校园卡可以去瀚林一号一楼usc服务中心，找到自助机器自己办理补办
5. 阿平平平安安: 谢谢！！！😭

共 5 条评论（主+子都混在一起）


In [12]:
print(result)

['肖迪种地and大敌: 应该得先在企微的“大学服务中心”申请补办校园卡，然后等通知到瀚林1号领。不过不用校园卡，直接用企微的热水服务也可以用热水的吧', '阿平平平安安: 谢谢uu🥰你是一个很好很好的宝宝', 'N/A: 那也得看他当初有没有把卡绑到过手机上', '十栖: 珠海校区补办校园卡可以去瀚林一号一楼usc服务中心，找到自助机器自己办理补办', '阿平平平安安: 谢谢！！！😭']


清洗文本

In [13]:
def clean_text(text):
    """清洗文本：移除多余换行和空格"""
    if not text:
        return ""
    # 多个换行变成一个
    text = re.sub(r'\n+', '\n', text)
    # 多个空格变成一个
    text = re.sub(r' +', ' ', text)
    return text.strip()

# 测试
dirty_text = "  这是  一段   有很多空格的\n\n\n文本  "
print(clean_text(dirty_text))

这是 一段 有很多空格的
文本


单个帖子的结构化处理

In [15]:
def process_post(post):
    """处理单个帖子，返回结构化文本"""
    detail = post.get('detail', {})
    
    # 提取核心字段（过滤掉手机号、头像等噪音）
    title = clean_text(detail.get('Title', ''))
    content = clean_text(detail.get('Content', ''))
    username = detail.get('UserName', '')
    post_time = format_timestamp(detail.get('PostTime', ''))
    
    # 提取评论
    comments = post.get('comments', [])
    comment_texts = extract_all_comments_flat(comments)
    
    # 组装成结构化文本（你的模板）
    formatted_text = f"标题: {title}\n"
    formatted_text += f"发布时间: {post_time}\n"
    formatted_text += f"作者: {username}\n"
    formatted_text += f"内容: {content}"
    
    # 如果有评论，添加到文本中
    if comment_texts:
        formatted_text += f"\n评论区: [{' | '.join(comment_texts)}]"
    
    return {
        'post_id': post.get('post_id'),
        'formatted_text': formatted_text,
        'metadata': {
            'title': title,
            'author': username,
            'post_time': post_time,
            'comment_count': len(comments)
        }
    }

# 处理第一条帖子测试
result = process_post(data[0])
print(result['formatted_text'])

标题: 校园集市上看到的，我有相同疑问
发布时间: 2026年03月26日
作者: iop
内容: “软工求助！
请问一下软工25级等级制相关问题～
rt，想问一下各位uu软院现在25级绩点制的话是不是会有很多人有相同的排名呀，那这样的话综测和推免要怎么算捏？
感谢各位uu！”
评论区: [miles Edgeworth: 大一综测不知道。后面课越来越多，到推免的时候排名一样的人没有想象的那么多吧]


In [17]:
print(type(result))

<class 'dict'>


In [18]:
# 处理所有帖子
cleaned_data = []

for post in data:
    processed = process_post(post)
    cleaned_data.append(processed)

print(f"处理完成，共 {len(cleaned_data)} 条帖子")

# 查看前3条
for i, item in enumerate(cleaned_data[:3], 1):
    print(f"\n{'='*60}")
    print(f"【帖子 {i}】")
    print(item['formatted_text'][:500] + "...")

处理完成，共 3401 条帖子

【帖子 1】
标题: 校园集市上看到的，我有相同疑问
发布时间: 2026年03月26日
作者: iop
内容: “软工求助！
请问一下软工25级等级制相关问题～
rt，想问一下各位uu软院现在25级绩点制的话是不是会有很多人有相同的排名呀，那这样的话综测和推免要怎么算捏？
感谢各位uu！”
评论区: [miles Edgeworth: 大一综测不知道。后面课越来越多，到推免的时候排名一样的人没有想象的那么多吧]...

【帖子 2】
标题: 【开发招募】软件工程学院本科生综合素质测评系统
发布时间: 2026年03月26日
作者: 超级小爱
内容: 开发需求：
目标：开发程序（或多维表格），通过在程序系统导入和填报数据（表格），由系统自动完成学院本科生每学年综合素质测评工作。
已有数据：历年学院本科生综合素质测评数据，包含学生参评个人基础信息、学年学业平均绩点、学生综测加分等表格，学年评选结果（人工评选的标准答案） ，学院本科生综合素质测评实施方案 以及 评选规则标准（用于测试程序逻辑是否正确）等。
要求：在用往年数据测试程序逻辑正确的基础上，导入和填报 今年 及 后续年份的学院本科生综合素质测评数据（仅管理员可见），能输出正确的当学年学院本科生综测排名 及 对应的中大优秀学生奖学金评选结果。
补充说明：
1. 导入数据为当学年学生参评个人基础信息、是否有不及格科目、学年学业平均绩点、学分完成情况等。
2. 学生自行填报数据为当学年每一项综测加分 及 对应的证明材料，由系统开通具有权限的账号，提交给班级综测评审小组审核。
其他需求：
人员：1人 推荐大二及以上，对相关工作有热情和责任心
DDL：7月前完成初版并进行...

【帖子 3】
标题: 【开发招募】软件工程学院AI迎新Agent
发布时间: 2026年03月26日
作者: 超级小爱
内容: 一、项目概述
1.1 项目背景
每年新生入学阶段，新生面临报到流程、宿舍管理、校园服务等各类疑问，传统答疑方式（人工、手册、微信群答疑）存在响应不及时、重复答疑、答疑时段受限等问题。为了解决以上问题，并结合软件工程学院学科优势，拟用AI解决上述问题。
1.2项目目标
总体目标：开发AI问答Agent网站（类似于豆包、千问、kimi等大模型Web网站），面向学院新生迎新场景，通过外挂新生

In [20]:
print(type(cleaned_data))

<class 'list'>


In [21]:
with open("cleaned_result.json", 'w', encoding='utf-8') as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=2)